# Week 8 Exercise: Smart Gadget Price Checker


## Goal
User pastes a gadget description and listing price. Agents estimate fair value and return buy/no-buy verdict.

## Concepts used
- `MarketResearchAgent` (Frontier-style RAG with vector store)
- `GadgetValuationSpecialist` (specialist second opinion)
- `DealDecisionPlanner` (autonomous tool-calling orchestration)
- OpenAI + Gradio


In [ ]:
import os
import re
import json
import chromadb
import gradio as gr
from openai import OpenAI
from sentence_transformers import SentenceTransformer

# Ensure OPENAI_API_KEY exists in your environment before running.
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


In [ ]:
# Tiny gadget knowledge base for RAG (lightweight)
seed_items = [
    {
        "name": "Apple iPhone 14 Pro Max 256GB",
        "description": "Refurbished Apple iPhone 14 Pro Max 256GB, 5G smartphone, premium camera.",
        "price": 899.0,
    },
    {
        "name": "Samsung Galaxy S23 Ultra 256GB",
        "description": "Samsung flagship smartphone with S Pen, high-end camera, AMOLED display.",
        "price": 949.0,
    },
    {
        "name": "Google Pixel 8 Pro 128GB",
        "description": "Google Pixel phone with advanced AI camera and clean Android experience.",
        "price": 799.0,
    },
    {
        "name": "Apple Watch Series 9 GPS",
        "description": "Apple smartwatch with fitness tracking, notifications, and health sensors.",
        "price": 349.0,
    },
    {
        "name": "Samsung Galaxy Watch 6 Classic",
        "description": "Samsung smartwatch with rotating bezel, health tracking, and LTE options.",
        "price": 299.0,
    },
    {
        "name": "Sony WH-1000XM5",
        "description": "Premium noise-canceling wireless over-ear headphones.",
        "price": 329.0,
    },
    {
        "name": "Bose QuietComfort Ultra",
        "description": "High-end ANC headphones with spatial audio and comfort fit.",
        "price": 379.0,
    },
    {
        "name": "Apple iPad Air 5th Gen",
        "description": "Lightweight tablet with M1 chip, ideal for media and productivity.",
        "price": 599.0,
    },
]

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("gadget_prices_lite")

# Reset if rerunning the notebook cell
existing = collection.get()
if existing and existing.get("ids"):
    collection.delete(ids=existing["ids"])

documents = [f"{x['name']}. {x['description']}" for x in seed_items]
ids = [f"item_{i}" for i in range(len(seed_items))]
metadatas = [{"price": x["price"], "name": x["name"]} for x in seed_items]
embeddings = embedder.encode(documents).tolist()

collection.add(ids=ids, documents=documents, metadatas=metadatas, embeddings=embeddings)
print(f"Loaded {len(seed_items)} items into local vector store.")


In [ ]:
class MarketResearchAgent:
    """RAG-based market researcher using vector search + OpenAI."""

    def __init__(self, collection, embedder, client, model="gpt-4o-mini"):
        self.collection = collection
        self.embedder = embedder
        self.client = client
        self.model = model

    def retrieve_similar_items(self, description: str, k: int = 3):
        query_embedding = self.embedder.encode([description]).tolist()
        results = self.collection.query(query_embeddings=query_embedding, n_results=k)
        docs = results["documents"][0]
        prices = [float(m["price"]) for m in results["metadatas"][0]]
        return docs, prices

    def _extract_number(self, text: str) -> float:
        clean = text.replace("$", "").replace(",", "")
        match = re.search(r"[-+]?\d*\.\d+|\d+", clean)
        return float(match.group()) if match else 0.0

    def estimate_price(self, description: str) -> float:
        docs, prices = self.retrieve_similar_items(description)
        context = "\n\n".join(
            [f"Similar item: {d}\nObserved price: ${p:.2f}" for d, p in zip(docs, prices)]
        )
        prompt = (
            "Estimate fair market price for this gadget. "
            "Return only a number.\n\n"
            f"Product: {description}\n\n"
            f"RAG context:\n{context}"
        )
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        return self._extract_number(response.choices[0].message.content or "0")


class GadgetValuationSpecialist:
    """Second-opinion specialist focused on gadget pricing."""

    def __init__(self, client, model="gpt-4o-mini"):
        self.client = client
        self.model = model

    def _extract_number(self, text: str) -> float:
        clean = text.replace("$", "").replace(",", "")
        match = re.search(r"[-+]?\d*\.\d+|\d+", clean)
        return float(match.group()) if match else 0.0

    def estimate_price(self, description: str) -> float:
        prompt = (
            "You are a gadget price specialist. "
            "Estimate fair market price in USD and return only a number.\n\n"
            f"Product: {description}"
        )
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        return self._extract_number(response.choices[0].message.content or "0")


In [ ]:
class DealDecisionPlanner:
    """Tool-calling planner that orchestrates researcher + specialist agents."""

    def __init__(self, market_research_agent, valuation_specialist, client, model="gpt-4o-mini"):
        self.market_researcher = market_research_agent
        self.valuation_specialist = valuation_specialist
        self.client = client
        self.model = model

        self.tools = [
            {
                "type": "function",
                "function": {
                    "name": "retrieve_similar_items",
                    "description": "Retrieve similar products and prices from vector store",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "description": {"type": "string"}
                        },
                        "required": ["description"],
                        "additionalProperties": False,
                    },
                },
            },
            {
                "type": "function",
                "function": {
                    "name": "estimate_price_frontier",
                    "description": "Estimate price using MarketResearchAgent with RAG",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "description": {"type": "string"}
                        },
                        "required": ["description"],
                        "additionalProperties": False,
                    },
                },
            },
            {
                "type": "function",
                "function": {
                    "name": "estimate_price_specialist",
                    "description": "Estimate price using GadgetValuationSpecialist",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "description": {"type": "string"}
                        },
                        "required": ["description"],
                        "additionalProperties": False,
                    },
                },
            },
            {
                "type": "function",
                "function": {
                    "name": "final_recommendation",
                    "description": "Create final recommendation from listing and estimates",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "listing_price": {"type": "number"},
                            "frontier_estimate": {"type": "number"},
                            "specialist_estimate": {"type": "number"},
                        },
                        "required": [
                            "listing_price",
                            "frontier_estimate",
                            "specialist_estimate",
                        ],
                        "additionalProperties": False,
                    },
                },
            },
        ]

    def retrieve_similar_items(self, description: str):
        docs, prices = self.market_researcher.retrieve_similar_items(description)
        return {
            "similar_items": docs,
            "prices": prices,
        }

    def estimate_price_frontier(self, description: str):
        return self.market_researcher.estimate_price(description)

    def estimate_price_specialist(self, description: str):
        return self.valuation_specialist.estimate_price(description)

    def final_recommendation(self, listing_price: float, frontier_estimate: float, specialist_estimate: float):
        estimated_value = round((frontier_estimate * 0.6) + (specialist_estimate * 0.4), 2)
        verdict = "Good deal" if listing_price < estimated_value else "Bad deal"
        return {
            "estimated_value": estimated_value,
            "listing_price": round(float(listing_price), 2),
            "verdict": verdict,
        }

    def _call_tool(self, name: str, args: dict):
        try:
            if name == "retrieve_similar_items":
                return self.retrieve_similar_items(**args)
            if name == "estimate_price_frontier":
                return self.estimate_price_frontier(**args)
            if name == "estimate_price_specialist":
                return self.estimate_price_specialist(**args)
            if name == "final_recommendation":
                return self.final_recommendation(**args)
        except Exception as e:
            return {"error": str(e), "tool": name}
        return {}

    def run(self, description: str, listing_price: float):
        messages = [
            {
                "role": "system",
                "content": (
                    "You are a planner. Use tools in this order: "
                    "retrieve_similar_items, estimate_price_frontier, estimate_price_specialist, final_recommendation. "
                    "After tools, respond with compact JSON keys: estimated_value, listing_price, verdict."
                ),
            },
            {
                "role": "user",
                "content": f"description: {description}\nlisting_price: {listing_price}",
            },
        ]

        for _ in range(6):
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                tools=self.tools,
                temperature=0,
            )
            msg = response.choices[0].message

            if msg.tool_calls:
                messages.append(
                    {
                        "role": "assistant",
                        "content": msg.content or "",
                        "tool_calls": [
                            {
                                "id": tc.id,
                                "type": "function",
                                "function": {
                                    "name": tc.function.name,
                                    "arguments": tc.function.arguments,
                                },
                            }
                            for tc in msg.tool_calls
                        ],
                    }
                )
                for tool_call in msg.tool_calls:
                    tool_name = tool_call.function.name
                    args = json.loads(tool_call.function.arguments or "{}")
                    result = self._call_tool(tool_name, args)
                    messages.append(
                        {
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "content": json.dumps(result),
                        }
                    )
            else:
                content = (msg.content or "").strip()
                try:
                    return json.loads(content)
                except json.JSONDecodeError:
                    # Safe fallback
                    f = self.estimate_price_frontier(description)
                    s = self.estimate_price_specialist(description)
                    return self.final_recommendation(listing_price, f, s)

        f = self.estimate_price_frontier(description)
        s = self.estimate_price_specialist(description)
        return self.final_recommendation(listing_price, f, s)


In [ ]:
market_research_agent = MarketResearchAgent(collection=collection, embedder=embedder, client=client)
valuation_specialist = GadgetValuationSpecialist(client=client)
planner = DealDecisionPlanner(market_research_agent, valuation_specialist, client)

# Quick test
sample_description = "Used iPhone 14 Pro Max 256GB in excellent condition, unlocked, 90% battery health"
sample_listing = 720
print(planner.run(sample_description, sample_listing))


In [ ]:
def smart_gadget_price_checker(description: str, listing_price: float):
    result = planner.run(description, float(listing_price))
    estimated = float(result.get("estimated_value", 0.0))
    listing = float(result.get("listing_price", listing_price))
    verdict = result.get("verdict", "Bad deal")
    return f"Estimated value: ${estimated:.2f}, Listing: ${listing:.2f}, Verdict: {verdict}"

ui = gr.Interface(
    fn=smart_gadget_price_checker,
    inputs=[
        gr.Textbox(label="Product description", lines=5),
        gr.Number(label="Listing price (USD)", value=500),
    ],
    outputs=gr.Textbox(label="Result"),
    title="Smart Gadget Price Checker",
    description="Paste a gadget description + listing price to get fair value and buy verdict.",
)

ui.launch()
